# Refined Stage 3D - granularity at matched lead time (RQ1)

**This is the notebook that removes the confound**, and the one an examiner will look for.

In the original Stage 2, granularity and horizon co-varied perfectly: native forecast 48
steps, hourly 24, daily 14. When daily looked better, there was no way to tell whether that
was aggregation helping or simply a shorter horizon.

Fixing horizon *in steps* does not solve it either: 24 steps is 24 minutes of wind and 24
days of daily data. Same number, still not the same question.

So the design is stated in **physical time**. History duration and lead time are held
constant; the step counts fall out of the metering rate:

```
context_steps = history_duration / step_minutes
horizon_steps = lead_time        / step_minutes
```

Now granularity is a genuine treatment: *the same question, the same history, different
representation.*

### The staircase

Some combinations are physically unreachable, and that is itself a result. At 1-minute
metering a 24-hour lead time needs 1,440 output steps — far beyond what these models emit
directly — and a 2,048-step context is only 1.4 days of history. Fine granularity buys
resolution and costs reach. That argument holds before any model runs and does not depend
on any metric.

### Why the history grid says 40 days

The models accept a context between 32 steps (TimesFM patches in 32s) and 2,048 steps. A
single history *duration* therefore only fits two granularities at once if their step sizes
differ by less than 64×. That is a hard arithmetic constraint, and it decides the grid:

| history | 30-min steps | hourly steps | daily steps | fits? |
|---|---|---|---|---|
| 7 d | 336 | 168 | 7 | daily below the 32-step floor |
| 30 d | 1,440 | 720 | 30 | daily below the floor **by two steps** |
| **40 d** | **1,920** | **960** | **40** | **all three fit** |
| 90 d | 4,320 | 2,160 | 90 | 30-min and hourly above the cap |

The first version of this notebook used 30 days, which put daily granularity one rung below
TimesFM's floor and quietly removed daily from every comparison — in a notebook whose entire
purpose is comparing granularities. 40 days is the smallest round figure where a 30-minute,
an hourly and a daily representation of the *same* history all fit, so `load` gets a genuine
three-way comparison.

Solar and wind cannot get one at any history, because 10-minute-to-daily is 144× and
1-minute-to-daily is 1,440×. For those, native-vs-hourly and hourly-vs-daily are the only
comparisons the models' context limits permit. That is a finding about deployment, not a
gap in the experiment: **at fine metering there is no single history window that can be
represented at both extremes of the granularity range.**

A 48-hour lead is included for the same reason — it is the shortest lead that gives daily
data more than a single output step.

> ### Why this notebook was rebuilt
>
> The first version fixed `N_WINDOWS = 5`. Because this design deliberately varies the step
> count with granularity, a fixed window count made the evaluated duration vary by a factor
> of 30 *within a single row of the comparison table* — the exact thing the notebook exists
> to prevent. Window counts are now planned per cell so that every granularity in a
> comparison is judged over a comparable stretch of real time.
>
> The previous version also loaded all three models once per feasible (lead, history, cell)
> combination. Windows are now built first and scored in one pass.

> **One load, not one load per configuration.** Every window for every configuration is
> built first, then scored in a single pass. `C.run()` holds one model in memory at a time
> and iterates the whole list, so this notebook loads three models in total. The earlier
> pattern -- `C.run()` inside the sweep loop -- reloaded all three on every iteration,
> which in 3H meant 144 load/free cycles to support 27 minutes of inference.

In [1]:
import sys, os, shutil, warnings, importlib
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import refined_stage_common as C

# A kernel that imported an older refined_stage_common keeps serving it from cache, and
# the failure surfaces much later as a missing column. Reload, then verify.
importlib.reload(C)
REQUIRED = "2026.09.10-relmae"
assert getattr(C, "__version__", None) == REQUIRED, (
    f"stale refined_stage_common (got {getattr(C, '__version__', 'none')}, "
    f"need {REQUIRED}) -- restart the kernel and Run All")
assert "plan_windows" in getattr(C, "__features__", set()), (
    "refined_stage_common predates the window planner -- restart the kernel and Run All")

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
PALETTE = {"chronos": "#2563eb", "timesfm": "#059669", "moirai": "#d97706"}
print(f"common module {C.__version__} from {C.__file__}")

common module 2026.09.10-relmae from e:\Thesis\EnergyForcastModel\refined_stage_common.py


In [2]:
DATA = C.get_data()

LEADS   = {"1 h": 60, "6 h": 360, "24 h": 1440, "48 h": 2880}   # minutes ahead
HISTORY = {"1 d": 1440, "7 d": 10080, "40 d": 57600}            # minutes of history
MAX_CONTEXT = 2048
MAX_HORIZON = 64        # Chronos-Bolt emits 64 steps directly; beyond that is autoregression
MIN_WINDOWS, MIN_DAYS, MAX_WINDOWS = 30, 14.0, 90
MIN_CTX = C.min_context_for(("chronos", "timesfm", "moirai"))

rows = []
for et in C.BASE:
    for gk in C.GRANULARITIES:
        for ln, lm in LEADS.items():
            for hn, hm in HISTORY.items():
                ok, c, h, why = C.feasible(et, gk, hm, lm, MAX_CONTEXT, MAX_HORIZON,
                                           min_context=MIN_CTX)
                rows.append({"etype": et, "gran": gk, "lead": ln, "history": hn,
                             "ok": ok, "context_steps": c, "horizon_steps": h,
                             "reason": why})
mask = pd.DataFrame(rows)
print("Feasible combinations:", int(mask.ok.sum()), "of", len(mask))
print()
print(mask.pivot_table(index=["etype", "gran"], columns=["lead", "history"],
                       values="ok", aggfunc="first").to_string())
mask.to_csv("refined_stage3d_feasibility.csv", index=False)

loaded cached panel <- refined_panel_cache.pkl
  load   5 series, 230,736–232,272 points
  solar  5 series, 52,560–52,560 points
  wind   5 series, 434,876–434,876 points
Feasible combinations: 44 of 108

lead            1 h                 24 h                 48 h                  6 h              
history         1 d   40 d    7 d    1 d   40 d    7 d    1 d   40 d    7 d    1 d   40 d    7 d
etype gran                                                                                      
load  1D      False  False  False  False   True  False  False   True  False  False  False  False
      1h      False   True   True  False   True   True  False   True   True  False   True   True
      native   True   True   True   True   True   True  False  False  False   True   True   True
solar 1D      False  False  False  False   True  False  False   True  False  False  False  False
      1h      False   True   True  False   True   True  False   True   True  False   True   True
      native   True

### Why each infeasible cell is infeasible state this in the thesis, it is a finding

In [3]:
print(mask[~mask.ok].groupby(["etype", "gran", "reason"]).size()
          .rename("cells").to_string())

etype  gran    reason                                                                        
load   1D      1 steps < minimum context 32 (TimesFM needs >= 32: it patches context in 32s)     2
               7 steps < minimum context 32 (TimesFM needs >= 32: it patches context in 32s)     2
               lead time shorter than one step                                                   6
       1h      24 steps < minimum context 32 (TimesFM needs >= 32: it patches context in 32s)    4
       native  96 steps > horizon cap 64                                                         3
solar  1D      1 steps < minimum context 32 (TimesFM needs >= 32: it patches context in 32s)     2
               7 steps < minimum context 32 (TimesFM needs >= 32: it patches context in 32s)     2
               lead time shorter than one step                                                   6
       1h      24 steps < minimum context 32 (TimesFM needs >= 32: it patches context in 32s)    4
       native  

## The window plan

The comparison this notebook makes is *across granularities within a (lead, history) row*.
So the thing that has to be comparable is the evaluated duration **within a row**, and that
is what the `evaluated` column is for. A row where one granularity covers 14 days and
another covers 2 hours is not a fair comparison, whatever the metric says.

In [4]:
import shutil
def archive(path, tag):
    old = path.replace(".csv", f"_{tag}.csv")
    if os.path.exists(path) and not os.path.exists(old):
        shutil.copy2(path, old); print(f"archived previous result -> {old}")

plan_rows = []
for r in mask[mask.ok].itertuples():
    d_min = C.step_minutes(r.etype, r.gran)
    shortest = min(len(v) for v in C.panel(DATA, r.etype, r.gran))
    want = C.windows_for(r.etype, r.gran, r.horizon_steps,
                         MIN_WINDOWS, MIN_DAYS, MAX_WINDOWS)
    supply = (shortest - r.context_steps) // r.horizon_steps
    n = int(min(want, supply))
    ev = n * r.horizon_steps * d_min
    plan_rows.append({"etype": r.etype, "gran": r.gran, "lead": r.lead,
                      "history": r.history, "context_steps": r.context_steps,
                      "horizon_steps": r.horizon_steps, "wanted": want,
                      "series_supports": int(supply), "n_windows": n,
                      "evaluated": C.human_duration(ev), "evaluated_min": ev,
                      "meets_floors": bool(n >= MIN_WINDOWS and ev >= MIN_DAYS * 1440)})
PLAN = pd.DataFrame(plan_rows)
PLAN = PLAN[PLAN.n_windows >= 1].reset_index(drop=True)
PLAN = PLAN.astype({"context_steps": int, "horizon_steps": int})
print(PLAN.drop(columns=["evaluated_min"]).to_string(index=False))

print("\nEvaluated duration spread WITHIN each (lead, history) row — this is the number "
      "that must not vary wildly, and the one the 5-window version got wrong:")
spread = (PLAN.groupby(["etype", "lead", "history"])["evaluated_min"]
              .agg(["min", "max", "count"]))
spread["ratio"] = (spread["max"] / spread["min"]).round(1)
spread["min"] = spread["min"].map(C.human_duration)
spread["max"] = spread["max"].map(C.human_duration)
print(spread[spread["count"] > 1].to_string())
PLAN.to_csv("refined_stage3d_window_plan.csv", index=False)

etype   gran lead history  context_steps  horizon_steps  wanted  series_supports  n_windows evaluated  meets_floors
 load native  1 h     1 d             48              2      90           115344         90     3.8 d         False
 load native  1 h     7 d            336              2      90           115200         90     3.8 d         False
 load native  1 h    40 d           1920              2      90           114408         90     3.8 d         False
 load native  6 h     1 d             48             12      56            19224         56    14.0 d          True
 load native  6 h     7 d            336             12      56            19200         56    14.0 d          True
 load native  6 h    40 d           1920             12      56            19068         56    14.0 d          True
 load native 24 h     1 d             48             48      30             4806         30    30.0 d          True
 load native 24 h     7 d            336             48      30         

## Build every window, then score once

In [5]:
archive("refined_stage3d_fixed_lead_time.csv", "5window")

# (lead, history) is recoverable from the window metadata -- lead_minutes and
# history_minutes are already written by Window.meta() -- so the whole grid can be scored
# in one pass instead of one C.run() per combination.
LEAD_OF = {v: k for k, v in LEADS.items()}
HIST_OF = {v: k for k, v in HISTORY.items()}

ALL = []
for r in PLAN.itertuples():
    W = C.build_windows(DATA, context_steps=r.context_steps,
                        horizon_steps=r.horizon_steps, n_windows=r.n_windows,
                        etypes=[r.etype], granularities=[r.gran], verbose=False)
    if not W:
        print(f"  {r.etype}/{r.gran} lead {r.lead} hist {r.history}: "
              f"series too short — skipped"); continue
    ALL.extend(W)

ALL.sort(key=lambda w: (w.context_steps, w.horizon_steps))   # keeps Moirai's cache warm
pairs = len({(w.context_steps, w.horizon_steps) for w in ALL})
print(f"\n{len(ALL):,} windows across {pairs} (context, horizon) pairs "
      f"-> {len(ALL)*3:,} forecasts, 3 model loads.")
print(f"   the old pattern would have loaded {pairs*3} times for the same work.")

# Cost per forecast is roughly linear in context length and barely depends on horizon.
# These coefficients are least-squares fits to the measured infer_sec of the 3H run on
# THIS machine, so the estimate is calibrated rather than generic. TimesFM is ~15x the
# cost of the other two at 2,048 steps and dominates the total.
COST = {"chronos": (0.0210, 1.94e-5), "moirai": (0.0286, 3.10e-5),
        "timesfm": (0.0952, 3.18e-4)}
est = {m: sum(a + b * w.context_steps for w in ALL) / 60 for m, (a, b) in COST.items()}
print("\n   projected inference time (from this machine's 3H timings):")
for m, v in sorted(est.items(), key=lambda kv: -kv[1]):
    print(f"     {m:8s} {v:6.0f} min")
print(f"     {'TOTAL':8s} {sum(est.values()):6.0f} min  + ~5 min of model loading")

MODELS = C.make_models(("chronos", "timesfm", "moirai"))
MODELS["moirai"].max_cached = 48    # one forecaster per (horizon, context); they share the
                                    # loaded module, so caching them costs almost no memory

sweep = C.run(MODELS, ALL)
sweep["lead"] = sweep.lead_minutes.map(LEAD_OF)
sweep["history"] = sweep.history_minutes.map(HIST_OF)
sweep["lead_min"] = sweep.lead_minutes
sweep["history_min"] = sweep.history_minutes
missing = sweep[sweep.lead.isna() | sweep.history.isna()]
assert missing.empty, f"{len(missing)} rows could not be mapped back to a (lead, history)"
C.save(sweep, "refined_stage3d_fixed_lead_time.csv")

archived previous result -> refined_stage3d_fixed_lead_time_5window.csv

11,630 windows across 24 (context, horizon) pairs -> 34,890 forecasts, 3 model loads.
   the old pattern would have loaded 72 times for the same work.

   projected inference time (from this machine's 3H timings):
     timesfm      56 min
     moirai        9 min
     chronos       6 min
     TOTAL        71 min  + ~5 min of model loading
loading chronos (max_context=1920, max_horizon=60) ...


chronos: 100%|██████████| 11630/11630 [03:09<00:00, 61.23it/s]


  chronos: 11630/11630 scored | 0.10 GB -> freeing
loading timesfm (max_context=1920, max_horizon=60) ...
    timesfm API = 2p5  (max_context=1920, max_horizon=60)


timesfm: 100%|██████████| 11630/11630 [2:26:49<00:00,  1.32it/s]     


  timesfm: 11630/11630 scored | 0.93 GB -> freeing
loading moirai (max_context=1920, max_horizon=60) ...


moirai: 100%|██████████| 11630/11630 [11:12<00:00, 17.30it/s]


  moirai: 11630/11630 scored | 0.06 GB -> freeing

NaN rate per model:
  chronos    0.0%
  moirai     0.0%
  timesfm    0.0%

All windows scored.
34,890 rows -> refined_stage3d_fixed_lead_time.csv  (8.50 MB)


'refined_stage3d_fixed_lead_time.csv'

## The de-confounded granularity comparison

In [6]:
cl = C.clean(sweep)
tab = (cl.groupby(["etype", "history", "lead", "gran_label", "model"])["relMAE"]
         .median().reset_index())
print(tab.pivot_table(index=["etype", "history", "lead", "model"],
                      columns="gran_label", values="relMAE").round(3).to_string())

30,285/34,890 windows used | 4605 degenerate excluded
gran_label                  daily  hourly  minutes (native)
etype history lead model                                   
load  1 d     1 h  chronos    NaN     NaN             0.893
                   moirai     NaN     NaN             0.840
                   timesfm    NaN     NaN             0.417
              24 h chronos    NaN     NaN             3.921
                   moirai     NaN     NaN             3.226
                   timesfm    NaN     NaN             1.158
              6 h  chronos    NaN     NaN             1.821
                   moirai     NaN     NaN             2.466
                   timesfm    NaN     NaN             0.988
      40 d    1 h  chronos    NaN   0.331             0.317
                   moirai     NaN   0.622             0.719
                   timesfm    NaN   0.282             0.227
              24 h chronos  0.610   0.737             0.855
                   moirai   0.683   1.053     

In [7]:
best = (cl.groupby(["etype", "history", "lead", "gran_label"])["relMAE"].median()
          .reset_index()
          .dropna(subset=["relMAE"]))          # some combos have no seasonal-naive benchmark

idx = best.groupby(["etype", "history", "lead"])["relMAE"].idxmin().dropna().astype(int)
pick = best.loc[idx]

print("Which granularity wins, at matched history and matched lead time:\n")
print(pick.rename(columns={"gran_label": "best_granularity", "relMAE": "median_relMAE"})
          .round(3).to_string(index=False))

# How many granularities were actually in contention for each row? A "winner" chosen from
# one candidate is not a winner.
n_cand = best.groupby(["etype", "history", "lead"]).size().rename("candidates")
print("\nGranularities compared per row (1 means no comparison was possible):")
print(n_cand.to_string())

miss = cl.groupby(["etype", "history", "lead"])["relMAE"].apply(lambda s: s.isna().all())
if miss.any():
    print(f"\n{int(miss.sum())} of {len(miss)} (type, history, lead) combinations had no "
          "relMAE at any granularity and were skipped:")
    for k in miss[miss].index.tolist():
        print("   ", k)

Which granularity wins, at matched history and matched lead time:

etype history lead best_granularity  median_relMAE
 load     1 d  1 h minutes (native)          0.647
 load     1 d 24 h minutes (native)          2.155
 load     1 d  6 h minutes (native)          1.595
 load    40 d  1 h           hourly          0.364
 load    40 d 24 h            daily          0.641
 load    40 d 48 h           hourly          0.797
 load    40 d  6 h           hourly          0.564
 load     7 d  1 h minutes (native)          0.381
 load     7 d 24 h           hourly          0.936
 load     7 d 48 h           hourly          0.872
 load     7 d  6 h           hourly          0.655
solar     1 d  1 h minutes (native)          0.838
solar     1 d  6 h minutes (native)          1.312
solar    40 d  1 h           hourly          0.854
solar    40 d 24 h            daily          0.764
solar    40 d 48 h            daily          0.778
solar    40 d  6 h           hourly          0.959
solar     7 d  

## Did the window count change this answer too?

3B showed that the 5-window design could flip the sign of a context effect. The same check
belongs here, because this table is the RQ1 headline.

In [3]:
# Runnable on its own from a cold kernel: it re-reads both CSVs from disk and imports
# what it needs, so the 161-minute inference above does not have to be repeated.
import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd
import refined_stage_common as C
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

# Only rows present in BOTH grids can testify about the window count. The history grid moved
# from 30 d to 40 d and gained a 48 h lead between the two runs (see the markdown above), so
# a plain outer join would count a grid change as a conclusion change. Self-contained on
# purpose: re-runnable without repeating the inference.
OLD = "refined_stage3d_fixed_lead_time_5window.csv"
NEW = "refined_stage3d_fixed_lead_time.csv"

def winners(path):
    x = C.clean(pd.read_csv(path), verbose=False)
    b = (x.groupby(["etype", "history", "lead", "gran_label"])["relMAE"].median()
           .reset_index().dropna(subset=["relMAE"]))
    i = b.groupby(["etype", "history", "lead"])["relMAE"].idxmin().dropna().astype(int)
    n = b.groupby(["etype", "history", "lead"]).size().rename("n_gran")
    return b.loc[i].merge(n.reset_index(), on=["etype", "history", "lead"])

if os.path.exists(OLD):
    op = winners(OLD).rename(columns={"gran_label": "old_best", "relMAE": "old_relMAE",
                                      "n_gran": "old_n_gran"})
    npw = winners(NEW).rename(columns={"gran_label": "new_best", "relMAE": "new_relMAE",
                                       "n_gran": "new_n_gran"})
    m = op.merge(npw, on=["etype", "history", "lead"], how="outer")
    m["in_both_grids"] = m.old_best.notna() & m.new_best.notna()
    m["changed"] = m.in_both_grids & (m.old_best != m.new_best)

    comp = m[m.in_both_grids]
    print("Rows the two runs have in common — the only ones that test the window count:\n")
    print(comp[["etype", "history", "lead", "old_best", "old_relMAE",
                "new_best", "new_relMAE", "new_n_gran", "changed"]]
          .round(3).to_string(index=False))
    print(f"\n{int(comp.changed.sum())} of {len(comp)} comparable rows pick a different "
          f"granularity once the window count is planned.")
    print(f"{len(m) - len(comp)} further rows exist in only one of the two grids "
          f"(30 d -> 40 d history, 48 h lead added) and say nothing about window count.")
    print("\nA row with new_n_gran == 1 had only one granularity in contention and is not "
          "a comparison at all — do not quote it as a granularity result.")
    m.to_csv("refined_stage3d_window_count_sensitivity.csv", index=False)
else:
    print(f"{OLD} not present — nothing to compare against.")

Rows the two runs have in common — the only ones that test the window count:

etype history lead         old_best  old_relMAE         new_best  new_relMAE  new_n_gran  changed
 load     1 d  1 h minutes (native)       1.078 minutes (native)       0.647         1.0    False
 load     1 d 24 h           hourly       1.816 minutes (native)       2.155         1.0     True
 load     1 d  6 h           hourly       1.667 minutes (native)       1.595         1.0     True
 load     7 d  1 h minutes (native)       0.429 minutes (native)       0.381         2.0    False
 load     7 d 24 h minutes (native)       1.006           hourly       0.936         2.0     True
 load     7 d  6 h           hourly       0.596           hourly       0.655         2.0    False
solar     1 d  6 h           hourly       1.057 minutes (native)       1.312         1.0     True
solar     7 d 24 h           hourly       1.078           hourly       1.079         1.0    False
solar     7 d  6 h minutes (native)     

Read the table above against the provisional ordering from `refined_stage2_baseline`.
Where they agree, the Stage 2 granularity claim survives the confound. Where they disagree,
**this notebook is the one to report** — and the disagreement is worth a paragraph, because
it is exactly the objection an examiner would raise.

Re-run `refined_stage3e_significance` and `refined_stage3f_selection_matrix` after this
notebook — both read `refined_stage3d_fixed_lead_time.csv`.

Next: `refined_stage3e_significance`.